#### Imports & Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# Lock randomness for exact reproducibility during viva evaluations
np.random.seed(42)

#### Data Ingestion, Quality Filtering & 80/20 Randomized Split

In [ ]:
# Load local datasets
file_1 = pd.read_excel(r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\MBP ControllerData 0521760 Overlock.xlsx")
file_2 = pd.read_excel(r"D:\Courses\IIT\AIDS\FYP\Preventing-Mechanism\PreProcessing\Synthetic Overlock Breakdowns.xlsx")

# Combine into a unified raw dataset
master_df = pd.concat([file_1, file_2], ignore_index=True)

# CORRUPTION FILTER: Strictly keep only rows where the vibration sequence starts with "10"
master_df = master_df[master_df['machineVibration'].astype(str).str.startswith('10')]

# Assign 'Healthy' to any row missing a specific breakdown label
master_df['Breakdown'] = master_df['Breakdown'].fillna('Healthy')

# Filter out any undocumented states
allowed_states = [
    "Healthy", "Needle Breakages", "High Foot Pressure", "Cut/Needle Hole", 
    "Thread Breakages", "Pneumatic Issues", "Thread Jamming", 
    "Code Uneven", "Roping", "Oil Mark", "Skip Stitches/Slip", 
    "Gathering/Puckering", "Waveness", "Binding/Seam Open", "Blade Broken"
]
master_df = master_df[master_df['Breakdown'].isin(allowed_states)].copy()

# Isolate classes to guarantee zero data leakage between normal and failure states
pure_healthy_df = master_df[master_df['Breakdown'] == 'Healthy'].copy()
pure_breakdown_df = master_df[master_df['Breakdown'] != 'Healthy'].copy()

# Shuffle the pure datasets to ensure a randomized distribution of machine states
pure_breakdown_df = pure_breakdown_df.sample(frac=1, random_state=42).reset_index(drop=True)
pure_healthy_df = pure_healthy_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Apply an exact 80/20 split to both subsets
split_h = int(len(pure_healthy_df) * 0.8)
split_b = int(len(pure_breakdown_df) * 0.8)

train_healthy = pure_healthy_df.iloc[:split_h]
test_healthy  = pure_healthy_df.iloc[split_h:]

train_breakdown = pure_breakdown_df.iloc[:split_b]
test_breakdown  = pure_breakdown_df.iloc[split_b:]

# Recombine into final Training and Testing DataFrames
train_df = pd.concat([train_healthy, train_breakdown], ignore_index=True)
test_df = pd.concat([test_healthy, test_breakdown], ignore_index=True)

print(f"Train Set: {len(train_healthy)} Healthy + {len(train_breakdown)} Breakdowns = {len(train_df)} total rows")
print(f"Test Set: {len(test_healthy)} Healthy + {len(test_breakdown)} Breakdowns = {len(test_df)} total rows")

#### Feature Extraction (Vibration Banding & Electrical Power)

In [ ]:
def extract_feature_df(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        if pd.notna(val):
            parts = str(val).split(',')
            try:
                # Dynamically parse sequential amplitude data into 10Hz frequency bands
                for i in range(0, len(parts)-1, 2):
                    f_start = int(parts[i])
                    f_end = f_start + 10
                    vib_dict[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
            except Exception:
                pass
        vib_records.append(vib_dict)
    
    vib_df = pd.DataFrame(vib_records)
    
    # Enforce exactly 60 frequency columns (10-20Hz up to 600-610Hz) and fill missing bands with 0
    expected_vib_cols = [f"{i}-{i+10}Hz" for i in range(10, 610, 10)]
    vib_df = vib_df.reindex(columns=expected_vib_cols, fill_value=0)
    
    # Extract electrical telemetry and forward-fill missing sensor readouts
    elec_df = df[[
        'machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
        'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax'
    ]].fillna(method='ffill').fillna(0)
    
    return pd.concat([vib_df, elec_df], axis=1)

X_train_df = extract_feature_df(train_df)
y_train_raw = train_df['Breakdown'].values

X_test_df = extract_feature_df(test_df)
y_test_raw = test_df['Breakdown'].values

# Convert to raw numpy arrays for the 3D sequence windowing function
X_train_raw = X_train_df.values
X_test_raw = X_test_df.values

print(f"Guaranteed Train Features: {X_train_raw.shape[1]}")
print(f"Guaranteed Test Features: {X_test_raw.shape[1]}")

#### Time-Series Sequencing (5-Step Windowing)

In [ ]:
TIME_STEPS = 5

def create_sequences(X, y, time_steps):
    """
    Transforms 2D tabular sensor rows into 3D sequential time windows 
    required for LSTM temporal pattern recognition.
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

X_train_seq_unscaled, y_train_seq = create_sequences(X_train_raw, y_train_raw, TIME_STEPS)
X_test_seq_unscaled, y_test_seq = create_sequences(X_test_raw, y_test_raw, TIME_STEPS)

#### Label Encoding & Leakage-Free Feature Scaling

In [ ]:
# Encode machine states into numeric categorical formats
encoder = LabelEncoder()
y_train = to_categorical(encoder.fit_transform(y_train_seq))
y_test = to_categorical(encoder.transform(y_test_seq))

# Standardize inputs to prevent dominant features (e.g., high voltage) from masking vibration signals
scaler = StandardScaler()

# Fit scaler strictly on training set to prevent data leakage, then transform
num_samples_train, _, num_features = X_train_seq_unscaled.shape
X_train = scaler.fit_transform(X_train_seq_unscaled.reshape(-1, num_features)).reshape(num_samples_train, TIME_STEPS, num_features)

# Apply previously fitted scaler to the test set
num_samples_test = X_test_seq_unscaled.shape[0]
X_test = scaler.transform(X_test_seq_unscaled.reshape(-1, num_features)).reshape(num_samples_test, TIME_STEPS, num_features)

print(f"Final Train Sequence Shape: {X_train.shape}")
print(f"Final Test Sequence Shape: {X_test.shape}")

#### LSTM Neural Network Architecture Definition

In [ ]:
num_classes = y_train.shape[1] 

model = Sequential()

# Layer 1: LSTM for temporal feature extraction 
model.add(LSTM(64, return_sequences=True, input_shape=(TIME_STEPS, num_features)))
model.add(Dropout(0.2)) # Prevents memorization of the training set
model.add(BatchNormalization())

# Layer 2: Deep LSTM layer to refine recognized sequence patterns
model.add(LSTM(32, return_sequences=False))
model.add(Dropout(0.2))

# Layer 3: Fully connected classification layer
model.add(Dense(32, activation='relu'))

# Output Layer: Softmax probability distribution across the 15 system states
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

#### Model Training Execution & Loss Visualization

In [ ]:
# Terminate training if validation loss degrades over 5 consecutive epochs
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Preserve only the optimal model iteration using Keras 3.0 standards (.keras)
checkpoint = ModelCheckpoint('best_overlock_lstm.keras', monitor='val_accuracy', save_best_only=True)

print("Executing LSTM Model Training...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# Plot training convergence for thesis evaluation
plt.figure(figsize=(10,4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('LSTM Model Loss over Epochs')
plt.legend()
plt.show()

#### Final Evaluation & Artifact Export for Live Inference

In [ ]:
# Export preprocessing artifacts required for real-time inference pipeline
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f)

# Output final system accuracy metric
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")